# Stage 1: Project setup
**Credit Risk Scoring & Loan Default Prediction**

**Business goal:** predict the probability that a loan applicant will have *payment difficulties*
(Home Credit's definition: the client was late by more than X days on at least one of the first Y instalments),
and turn that probability into an approve / refer / decline recommendation that balances credit losses
against lost business.

This notebook checks that the environment, folder structure and data are ready.

## Project architecture
```
credit-risk-scoring/
├── config/config.yaml          <- every path, split size and business assumption in one place
├── data/
│   ├── raw/                    <- Kaggle CSVs (never modified, never committed)
│   ├── interim/                <- cleaned application tables
│   ├── processed/              <- feature matrices, split ids, predictions
│   └── sample/                 <- optional synthetic data for smoke tests
├── database/home_credit.db     <- SQLite database built in stage 3
├── sql/                        <- views + named analysis queries
├── notebooks/                  <- 13 stage notebooks (run in order)
├── src/credit_risk/            <- reusable, tested Python package used by notebooks AND the app
├── models/                     <- fitted models, feature contract, decision policy
├── reports/                    <- figures, SQL results, metrics, data dictionary
├── app/streamlit_app.py        <- stage 13 dashboard
├── scripts/                    <- data download helper, synthetic data, notebook runner
└── tests/                      <- pytest unit tests for features and decision logic
```

**Why a `src/` package?** Logic lives in one importable place, so the notebooks stay readable and the
Streamlit app scores applicants with *exactly* the same feature code used in training. Keeping training
and serving consistent like this is the most common thing a model-risk reviewer checks.

## Environment setup (run once, in Anaconda Prompt / terminal)
```bash
conda env create -f environment.yml      # creates the 'credit-risk' env and installs src/ in editable mode
conda activate credit-risk
python -m ipykernel install --user --name credit-risk --display-name "Python (credit-risk)"
```
## Data download
1. Accept the competition rules at https://www.kaggle.com/competitions/home-credit-default-risk/rules
2. Put your Kaggle API token in `~/.kaggle/kaggle.json` (Kaggle → Settings → API → Create New Token)
3. `python scripts/download_data.py` (or download the zip manually and extract it into `data/raw/`)

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Library versions

In [2]:
import importlib
import platform

packages = ["pandas", "numpy", "pyarrow", "sklearn", "xgboost", "lightgbm", "shap", "fairlearn",
            "matplotlib", "seaborn", "plotly", "streamlit", "yaml", "joblib"]
print(f"Python {platform.python_version()} on {platform.system()}")
missing = []
for name in packages:
    try:
        mod = importlib.import_module(name)
        print(f"  {name:<12} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        missing.append(name)
        print(f"  {name:<12} MISSING")
assert not missing, f"Install missing packages: {missing}"

Python 3.14.6 on Windows
  pandas       3.0.3
  numpy        2.4.6
  pyarrow      23.0.1
  sklearn      1.9.0
  xgboost      3.4.1
  lightgbm     4.7.0
  shap         0.51.0
  fairlearn    0.15.0
  matplotlib   3.11.0
  seaborn      0.13.2
  plotly       6.7.0
  streamlit    1.58.0
  yaml         6.0.3
  joblib       1.5.3


## 2. Configuration and paths

In [3]:
import os

from credit_risk.config import get_paths, load_config

cfg = load_config()
paths = get_paths(cfg)
for k, v in paths.items():
    print(f"{k:<15} {v.relative_to(ROOT) if v.is_relative_to(ROOT) else v}")

if os.environ.get("CREDIT_RISK_RAW_DIR"):
    print("\nWARNING: CREDIT_RISK_RAW_DIR is set, so the notebooks will read", os.environ["CREDIT_RISK_RAW_DIR"],
          "\nIf that's the synthetic sample, results are for smoke-testing only.")

raw_data        C:\Users\AVNEET\Downloads\credit-risk-data\raw
interim_data    C:\Users\AVNEET\Downloads\credit-risk-data\interim
processed_data  C:\Users\AVNEET\Downloads\credit-risk-data\processed
database        C:\Users\AVNEET\Downloads\credit-risk-data\database\home_credit.db
models          C:\Users\AVNEET\Downloads\credit-risk-data\models
reports         reports
figures         reports\figures
sql_dir         sql


## 3. Raw data check

In [4]:
import pandas as pd

rows = []
for key, fname in cfg["data"]["files"].items():
    f = paths["raw_data"] / fname
    rows.append({"table": key, "file": fname, "present": f.exists(),
                 "size_mb": round(f.stat().st_size / 1024**2, 1) if f.exists() else None})
status = pd.DataFrame(rows)
display(status)
if not status["present"].all():
    print("Some files are missing. See the download instructions above.")

,table,file,present,size_mb
0,application_train,application_train.csv,True,158.4
1,application_test,application_test.csv,True,25.3
2,bureau,bureau.csv,True,162.1
3,bureau_balance,bureau_balance.csv,True,358.2
4,previous_application,previous_application.csv,True,386.2
5,pos_cash_balance,POS_CASH_balance.csv,True,374.5
6,installments_payments,installments_payments.csv,True,689.6
7,credit_card_balance,credit_card_balance.csv,True,404.9
8,columns_description,HomeCredit_columns_description.csv,True,0.0


## 4. Business assumptions used later
These drive the cost-sensitive decision framework in Stage 10. In an interview, say plainly that they
are assumptions and show how sensitive the decision is to them (we do that in Stage 10).

In [5]:
pd.Series(cfg["business"], name="value").to_frame()

,value
lgd,0.60
profit_margin,0.15
refer_band_ratio,0.70
